# Clase 8 — Aprendizaje por refuerzo para LLMs

**Laboratorio**

En la clase 7 ajustamos un modelo base con SFT y QLoRA para que imitara pares de instrucción y respuesta. En las diapositivas de hoy vimos cómo un LLM aprende, en cambio, de recompensas.

**Una distinción clave.** El SFT de la clase 7 necesitaba la respuesta correcta para cada prompt. Los métodos de hoy solo necesitan saber **qué respuesta es mejor**, y esa señal puede venir de distintas fuentes:

| Método | Datos que necesita | En este notebook |
|---|---|---|
| SFT (clase 7) | Pares prompt → respuesta ideal | — |
| RLHF | Prompts + un modelo de recompensa, entrenado con pares de preferencias | Secciones 1 y 4 |
| DPO | Tripletas (prompt, ganadora, perdedora), sin modelo de recompensa aparte | Sección 2 |
| RLVR | Prompts + un verificador: una regla que revisa la respuesta | Sección 3 |

El notebook tiene cuatro partes:

1. **La función de recompensa r(x, y):** primero la idea con Bradley-Terry y después un modelo de recompensa de verdad, entrenado con `RewardTrainer`.
2. **DPO** con `DPOTrainer`, y la recompensa implícita de Qwen.
3. **GRPO** con `GRPOTrainer` y un verificador (RLVR).
4. **RLHF de verdad:** GRPO con el modelo de recompensa como juez. Reward hacking y el freno KL.

Salvo el ejemplo de Bradley-Terry, que es una tabla de cinco números, todo se hace sobre un LLM de verdad: Qwen2.5-0.5B-Instruct con LoRA, entrenado con **TRL**, la librería de Hugging Face del `SFTTrainer` de la clase 7.

**En Colab:** usen una GPU (Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4). Cada entrenamiento tarda menos de un minuto con GPU; en CPU, varios minutos.

**Celdas interactivas.** Las celdas marcadas con 🎛️ son formularios: en Colab, cambien un campo o muevan un control y la celda se vuelve a ejecutar sola. Fuera de Colab, cambien los valores en el código y ejecuten la celda.

Como siempre: antes de ejecutar un experimento, anticipen el resultado.

In [ ]:
#@title Preparación: librerías y estilo de los gráficos
# Las mismas versiones de la tarea 3. Si falta TRL (por ejemplo, en Colab), se instala.
import importlib.util
if importlib.util.find_spec("trl") is None:
    %pip install -q transformers==5.22.1 trl==1.12.0 peft==0.21.2 datasets

# Colab trae un torchao antiguo que peft rechaza con un ImportError. El notebook no lo usa: se desinstala.
from importlib.metadata import PackageNotFoundError, version
try:
    if tuple(int(n) for n in version("torchao").split(".")[:2]) < (0, 16):
        %pip uninstall -q -y torchao
        importlib.invalidate_caches()
except PackageNotFoundError:
    pass

import logging, os, random, re, time, warnings

# transformers carga los pesos con varios hilos; en algunos equipos (Mac con GPU de Apple) eso se cuelga o se cae.
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import datasets
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, PrinterCallback, TrainerCallback
from transformers.utils import logging as hf_logging
from trl import DPOConfig, DPOTrainer, GRPOConfig, GRPOTrainer, RewardConfig, RewardTrainer

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); datasets.disable_progress_bars()
logging.getLogger("trl").setLevel(logging.ERROR)

%config InlineBackend.figure_formats = ['retina']
plt.rcParams.update({"figure.dpi": 100, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
AZUL, ROJO, VERDE, NARANJO, MORADO, GRIS = "#6c8ebf", "#b85450", "#82b366", "#d79b00", "#9673a6", "#999999"
torch.manual_seed(0)

MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠️ No hay GPU: cada entrenamiento tardará varios minutos en vez de segundos.\n"
          "   En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4, y vuelvan a ejecutar desde el inicio.")

def lora(tarea):
    """LoRA como en la clase 7: solo se entrenan las matrices A y B."""
    return LoraConfig(
        r=16,                          # rango de las matrices A y B
        lora_alpha=32,                 # escala del adapter (alpha / r = 2)
        target_modules="all-linear",   # un adapter en cada capa lineal del modelo
        task_type=tarea,               # "CAUSAL_LM" para generar texto, "SEQ_CLS" para dar un puntaje
    )

class Avance(TrainerCallback):
    """Muestra en una sola línea por qué paso va el entrenamiento."""
    def on_step_end(self, args, state, control, **kwargs):
        print(f"\rpaso {state.global_step} de {state.max_steps}", end="")

def entrenar(entrenador):
    """Entrena mostrando solo el avance; el registro de cada paso lo leemos después con registros()."""
    entrenador.remove_callback(PrinterCallback)   # sin esto, imprime una línea larga por cada registro
    entrenador.add_callback(Avance())
    inicio = time.time()
    entrenador.train()
    print(f"\rEntrenado en {time.time() - inicio:.0f} s      ")

def registros(entrenador, *claves):
    """Los valores que TRL fue registrando durante el entrenamiento: una lista por clave."""
    # log_history es una lista de diccionarios, uno cada logging_steps pasos: {"step": 2, "loss": 0.14, ...}
    filas = [l for l in entrenador.state.log_history if all(k in l for k in claves)]
    return [l["step"] for l in filas], *[[float(l[k]) for l in filas] for k in claves]

# 1. La función de recompensa r(x, y)

> **Datos:** pares de preferencias (prompt, ganadora, perdedora). **Para qué:** aprender una función r(x, y) que le ponga nota a cualquier respuesta. Es la recompensa del RLHF.

## 1.1 La idea en pequeño: Bradley-Terry

Una tienda quiere un modelo de recompensa para su asistente. Nadie sabe asignarle una nota absoluta a una respuesta, pero es fácil decir cuál de dos es mejor.

Simulemos esa situación: cinco respuestas candidatas a la misma pregunta, cada una con una **recompensa verdadera** que el modelo no conoce. Las personas comparan pares y eligen según Bradley-Terry:

$$P(y_i \succ y_j) = \sigma\big(r_i - r_j\big)$$

In [ ]:
pregunta = "¿Hacen envíos a regiones?"
respuestas = [
    "Sí, enviamos a todo Chile; a regiones tarda de 3 a 5 días hábiles.",
    "Sí, enviamos a regiones.",
    "Sí. Revise la página de despachos.",
    "Depende.",
    "Consulte nuestras políticas.",
]
r_verdadera = torch.tensor([2.0, 1.0, 0.3, -0.8, -1.5])

# 1.000 comparaciones entre pares al azar
n = 1000
i = torch.randint(0, 5, (n,))
j = (i + torch.randint(1, 5, (n,))) % 5          # j ≠ i
gana_i = torch.rand(n) < torch.sigmoid(r_verdadera[i] - r_verdadera[j])
ganadora = torch.where(gana_i, i, j)
perdedora = torch.where(gana_i, j, i)

print(f"Pregunta: {pregunta}\n")
for k, texto_r in enumerate(respuestas):
    partidas = ((i == k) | (j == k)).sum().item()
    print(f"[{k}] gana {(ganadora == k).sum().item() / partidas:4.0%} de sus comparaciones · {texto_r}")

Ahora ajustamos un modelo de recompensa: un número $r_k$ por respuesta, entrenado con la misma pérdida que se usa para los modelos de recompensa de verdad,

$$L = -\,\mathbb{E}\big[\log \sigma(r_{w} - r_{l})\big].$$

Es una regresión logística sobre pares. En un LLM real, $r$ es una red (el LLM con una cabeza lineal); aquí es una tabla de 5 números.

In [ ]:
r = torch.zeros(5, requires_grad=True)
opt = torch.optim.Adam([r], lr=0.05)
for paso in range(500):
    perdida = -F.logsigmoid(r[ganadora] - r[perdedora]).mean()
    opt.zero_grad(); perdida.backward(); opt.step()

r_estimada = r.detach()
print(f"pérdida final: {perdida.item():.3f}\n")
print("respuesta   verdadera (centrada)   estimada")
for k in range(5):
    print(f"   [{k}]          {r_verdadera[k] - r_verdadera.mean():+.2f}              {r_estimada[k] - r_estimada.mean():+.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(5)
ax.bar(x - 0.2, r_verdadera - r_verdadera.mean(), 0.4, color=GRIS, label="verdadera")
ax.bar(x + 0.2, r_estimada - r_estimada.mean(), 0.4, color=AZUL, label="estimada")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, [f"[{k}]" for k in x]); ax.set_ylabel("recompensa (centrada)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

Con 3.000 comparaciones recuperamos las recompensas casi exactas, **sin que nadie haya puesto una nota absoluta**.

Comparamos todo *centrado* (restando el promedio) porque Bradley-Terry solo ve diferencias: si sumamos una constante a todas las recompensas, la pérdida no cambia.

In [ ]:
for constante in [0, 10, -100]:
    p = -F.logsigmoid((r_estimada + constante)[ganadora] - (r_estimada + constante)[perdedora]).mean()
    print(f"recompensas + {constante:4d} → pérdida {p:.4f}")

## 1.2 Un modelo de recompensa de verdad, con TRL

El objetivo es tener una función r(x, y) que le ponga nota a cualquier respuesta. En un LLM real, r(x, y) no es una tabla: es **el mismo LLM con una cabeza lineal** que devuelve un número. Se entrena con la misma pérdida de arriba, $-\log\sigma(r_w - r_l)$, y TRL lo trae hecho: `RewardTrainer`.

Los datos son pares de preferencias de la tienda: para cada pregunta, la respuesta que una persona eligió (`chosen`) y la que rechazó (`rejected`). Aquí los generamos con plantillas; en un caso real salen de personas comparando respuestas. Es lo que hace ChatGPT cuando muestra dos respuestas al mismo prompt y pregunta cuál prefieres: cada clic es un par (prompt, ganadora, perdedora).

In [ ]:
CIUDADES = ["Arica", "Iquique", "Antofagasta", "La Serena", "Valparaíso", "Rancagua", "Talca", "Concepción", "Temuco", "Valdivia", "Puerto Montt", "Punta Arenas"]
PRODUCTOS = ["una mochila", "un teclado", "una lámpara", "unos audífonos", "una taza", "un termo", "una polera", "un cargador", "una agenda", "un paraguas", "una bufanda", "un cuaderno"]
PLANTILLAS = [
    ("¿Hacen envíos a {c}?", "Sí, enviamos a {c}. El despacho tarda de 3 a 5 días hábiles y el costo se calcula al pagar."),
    ("¿Cuánto demora un pedido a {c}?", "Un pedido a {c} tarda de 3 a 5 días hábiles. Te enviamos un código de seguimiento apenas sale de bodega."),
    ("Compré {p} y llegó con fallas, ¿qué hago?", "Lamento lo ocurrido. Envíanos una foto del producto y tu número de pedido, y gestionamos el cambio sin costo."),
    ("¿Puedo devolver {p} que compré la semana pasada?", "Sí, tienes 30 días para devolver el producto. Solo necesitas la boleta y que esté sin uso."),
    ("¿Tienen stock de {p}?", "Sí, tenemos stock. Si haces el pedido hoy, sale de bodega mañana."),
    ("¿Qué medios de pago aceptan para comprar {p}?", "Aceptamos tarjetas de débito y crédito, y transferencia. Con tarjeta de crédito puedes pagar hasta en 6 cuotas sin interés."),
]
RECHAZADAS = ["Consulte nuestras políticas.", "Depende.", "No sé.", "Revise la página web.", "Eso no lo vemos aquí.", "Llame al servicio al cliente.", "No es nuestro problema.", "Envíos."]

# 6 plantillas × 12 ciudades o productos = 72 pares. La rechazada es una respuesta cortante al azar.
azar = random.Random(0)
filas = []
for pregunta, elegida in PLANTILLAS:
    for c, p in zip(CIUDADES, PRODUCTOS):
        filas.append({   # las tres columnas que esperan RewardTrainer y DPOTrainer, en formato de chat
            "prompt": [{"role": "user", "content": pregunta.format(c=c, p=p)}],
            "chosen": [{"role": "assistant", "content": elegida.format(c=c, p=p)}],
            "rejected": [{"role": "assistant", "content": azar.choice(RECHAZADAS)}],
        })
azar.shuffle(filas)
preferencias = Dataset.from_list(filas[:60])        # para entrenar
preferencias_prueba = filas[60:]                    # 12 pares que el modelo no verá

print(len(preferencias), "pares de entrenamiento. Ejemplos:")
for f in filas[:3]:
    print(f"\n  prompt:    {f['prompt'][0]['content']}\n  elegida:   {f['chosen'][0]['content']}\n  rechazada: {f['rejected'][0]['content']}")

In [ ]:
# La configuración: cuánto y cómo se entrena. Casi todas las opciones son las de cualquier Trainer de Hugging Face.
config = RewardConfig(
    output_dir="rm",                   # carpeta de trabajo del entrenador
    max_steps=20,                      # 20 pasos de optimización
    per_device_train_batch_size=8,     # 8 pares (elegida, rechazada) por paso
    learning_rate=1e-4,
    logging_steps=2,                   # registra la pérdida y la exactitud cada 2 pasos
    save_strategy="no",                # no guarda puntos de control en disco
    report_to="none",                  # no envía los registros a servicios externos
    disable_tqdm=True,                 # sin barra de progreso
    seed=0,
)

# El entrenador junta el modelo, los datos y la configuración.
entrenador_rm = RewardTrainer(
    model=MODELO,                      # el nombre del modelo: TRL lo carga con una cabeza lineal de una salida, r(x, y)
    args=config,
    train_dataset=preferencias,        # columnas "prompt", "chosen" y "rejected"
    peft_config=lora("SEQ_CLS"),       # LoRA: solo se entrenan el adapter y la cabeza nueva
)
entrenar(entrenador_rm)                # minimiza −log σ(r_elegida − r_rechazada), la pérdida de Bradley-Terry

# "accuracy": fracción de pares del lote en que la elegida saca más nota que la rechazada
pasos, perdida, exactitud = registros(entrenador_rm, "loss", "accuracy")
print("\npaso   pérdida   ganadora con más nota")
for p, l, a in zip(pasos, perdida, exactitud):
    print(f"{p:>4}   {l:7.4f}   {a:.0%}")

La pérdida es la de Bradley-Terry y la **exactitud** es la fracción de pares en que la ganadora saca más nota que la perdedora. Con 60 pares y unos pocos pasos ya ordena bien los pares de entrenamiento.

Ahora usemos el modelo como lo que es: una función r(x, y). Primero en los 12 pares que no vio, y después en respuestas escritas por nosotros a una pregunta nueva. **Antes de ejecutar:** miren la cuarta respuesta, larga y amable pero vacía. ¿Qué nota creen que saca?

In [ ]:
# El modelo entrenado y el tokenizador que TRL cargó para él
modelo_rm, tok_rm = entrenador_rm.model.eval(), entrenador_rm.processing_class

@torch.no_grad()
def r(pregunta, respuesta):
    """La función de recompensa aprendida: un número por (prompt, respuesta)."""
    # el mismo formato del entrenamiento: la conversación completa, con la plantilla de chat
    texto = tok_rm.apply_chat_template([{"role": "user", "content": pregunta}, {"role": "assistant", "content": respuesta}], tokenize=False)
    entrada = tok_rm(texto, return_tensors="pt").to(modelo_rm.device)
    return modelo_rm(**entrada).logits[0, 0].item()      # la salida de la cabeza lineal: un solo número

# ¿la elegida saca más nota que la rechazada en los pares que no vio?
aciertos = [r(f["prompt"][0]["content"], f["chosen"][0]["content"]) > r(f["prompt"][0]["content"], f["rejected"][0]["content"]) for f in preferencias_prueba]
print(f"Pares de prueba bien ordenados: {sum(aciertos)} de {len(aciertos)}\n")

pregunta = "¿Hacen cambios de talla en poleras?"
candidatas = [
    "Sí, puedes cambiar la talla dentro de 30 días con tu boleta.",
    "No hacemos cambios de talla, lo siento.",
    "No sé.",
    "Gracias por escribirnos. Para nosotros es muy importante tu consulta y la atenderemos con el mayor de los gustos, porque nuestros clientes son lo primero.",
]
notas = [r(pregunta, c) for c in candidatas]
for n, c in zip(notas, candidatas):
    print(f"{n:+6.1f}   {c}")

plt.figure(figsize=(6, 2.6))
plt.barh(range(4), notas, color=[VERDE, GRIS, GRIS, NARANJO])
plt.yticks(range(4), ["útil", "negativa clara", "“No sé.”", "larga y vacía"]); plt.gca().invert_yaxis()
plt.axvline(0, color="black", lw=0.8); plt.xlabel("r(x, y)"); plt.tight_layout(); plt.show()

**🎛️ Su turno: engañen al juez.** Escriban una respuesta y miren qué nota le pone el modelo de recompensa. ¿Pueden conseguir más de +15 con una respuesta que no conteste la pregunta? ¿Qué tienen en común las que sacan buena nota?

In [ ]:
#@title 🎛️ Escriban una respuesta y miren su nota { run: "auto" }
mi_pregunta = "¿Hacen cambios de talla en poleras?" #@param {type:"string"}
mi_respuesta = "Estimado cliente, agradecemos mucho su preferencia y su paciencia." #@param {type:"string"}

print(f"{r(mi_pregunta, mi_respuesta):+6.1f}   {mi_respuesta}")
print(f"{r(mi_pregunta, 'No sé.'):+6.1f}   No sé.   (para comparar)")

Ya tenemos lo que el RL necesita: una función que le pone nota a cualquier respuesta, aprendida solo con comparaciones.

Pero miren la respuesta larga y vacía: no contesta la pregunta y saca una nota alta. En nuestros datos las elegidas eran siempre más largas y amables que las rechazadas, y el modelo aprendió ese atajo. Un modelo que maximice esta recompensa puede descubrirlo y explotarlo: es el **reward hacking** que vimos en clase, y lo vamos a provocar en la sección 4.

# 2. DPO: alinear sin aprendizaje por refuerzo

> **Datos:** tripletas (prompt, ganadora, perdedora). Ni modelo de recompensa ni generación durante el entrenamiento.

## 2.1 DPO con TRL

RLHF con PPO tiene dos fuentes de complejidad: el modelo de recompensa y el ciclo de RL (generar, poner nota y actualizar, miles de veces). **DPO elimina las dos**: con los mismos pares de preferencias entrena a la política directamente, con una pérdida supervisada.

La idea, antes de la fórmula: para cada par, DPO compara al modelo con la referencia. ¿Hace más probable la respuesta elegida que antes? ¿Y la rechazada? Entrena para que **la elegida suba y la rechazada baje**, siempre medido respecto de $\pi_{ref}$.

Esa medida es la **recompensa implícita** de una respuesta, $\beta\log\frac{\pi_\theta(y \mid x)}{\pi_{ref}(y \mid x)}$: cuánto más probable la hace el modelo que la referencia. La pérdida es la misma del modelo de recompensa de la sección 1, con esa nota en lugar de $r$:

$$L_{DPO} = -\,\mathbb{E}\Big[\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w \mid x)}{\pi_{ref}(y_w \mid x)} - \beta\log\frac{\pi_\theta(y_l \mid x)}{\pi_{ref}(y_l \mid x)}\Big)\Big].$$

Aquí $y_w$ es la elegida (`chosen` en TRL) e $y_l$ la rechazada (`rejected`). Usaremos **los mismos 60 pares** de la sección 1.2. Allá entrenaron un modelo de recompensa; aquí entrenan directamente a la política, con `DPOTrainer`.

Primero, cómo responde Qwen antes de entrenar, a tres preguntas que no están en los datos:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO)
politica = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(DISPOSITIVO)

PREGUNTAS_PRUEBA = ["¿Hacen envíos a Chillán?", "¿Puedo pagar con transferencia?", "Mi pedido llegó incompleto, ¿qué hago?"]

@torch.no_grad()
def responder(modelo, pregunta, max_tokens=60):
    # add_generation_prompt deja el texto listo para que el modelo escriba el turno del asistente
    entrada = tokenizer.apply_chat_template([{"role": "user", "content": pregunta}], add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True).to(modelo.device)
    salida = modelo.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)      # greedy: siempre la misma respuesta
    # la salida incluye el prompt: nos quedamos con los tokens nuevos
    return tokenizer.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).replace("\n", " ")

antes = [responder(politica, q) for q in PREGUNTAS_PRUEBA]
for q, a in zip(PREGUNTAS_PRUEBA, antes):
    print(f"{q}\n   → {a[:220]}\n")

Responde como un asistente genérico: largo, y a veces aclarando que es una inteligencia artificial. Los pares de preferencias dicen otra cosa: la tienda prefiere respuestas cortas, concretas y en su propia voz.

Con `peft_config`, TRL agrega LoRA y deja congelado el resto, como en la clase 7. Además, $\pi_{ref}$ no ocupa una segunda copia en memoria: es el mismo modelo con el adapter apagado.

In [ ]:
config = DPOConfig(
    output_dir="dpo",                  # carpeta de trabajo del entrenador
    max_steps=30,                      # 30 pasos de optimización
    per_device_train_batch_size=4,     # 4 pares por paso
    learning_rate=1e-4,                # tamaño de cada actualización de los pesos del adapter
    beta=0.1,                          # el β de la pérdida: más chico, más se puede alejar de la referencia
    logging_steps=2,                   # registra la pérdida y las recompensas implícitas cada 2 pasos
    save_strategy="no",                # no guarda puntos de control en disco
    report_to="none",                  # no envía los registros a servicios externos (como Weights & Biases)
    disable_tqdm=True,                 # sin barra de progreso: el avance lo muestra entrenar()
    seed=0,                            # semilla, para que el resultado se pueda repetir
)
entrenador_dpo = DPOTrainer(
    model=politica,                    # el LLM que queremos alinear (ya cargado)
    args=config,                       # la configuración de arriba
    train_dataset=preferencias,        # los mismos pares de la sección 1.2: "prompt", "chosen" y "rejected"
    processing_class=tokenizer,        # el tokenizador: aplica la plantilla de chat a cada par
    peft_config=lora("CAUSAL_LM"),     # se entrena el adapter; la referencia π_ref es el modelo con el adapter apagado
)
entrenar(entrenador_dpo)               # no genera texto: solo calcula log-probabilidades de las respuestas de los pares

# TRL registra la recompensa implícita β·log(π_θ/π_ref) de las elegidas y de las rechazadas
pasos, r_elegida, r_rechazada = registros(entrenador_dpo, "rewards/chosen", "rewards/rejected")
plt.figure(figsize=(6, 3))
plt.plot(pasos, r_elegida, color=VERDE, lw=2, label="elegidas")
plt.plot(pasos, r_rechazada, color=ROJO, lw=2, label="rechazadas")
plt.axhline(0, color="black", lw=0.8); plt.xlabel("paso de entrenamiento"); plt.ylabel("recompensa implícita")
plt.legend(frameon=False, loc="lower left"); plt.tight_layout(); plt.show()

La recompensa implícita parte en 0 para todas las respuestas (al inicio $\pi_\theta = \pi_{ref}$) y se separa: el modelo hace más probables las elegidas y menos probables las rechazadas. Es la figura de la diapositiva “Qué aprende DPO con cada par”. Nunca generó una respuesta ni consultó un modelo de recompensa: es un entrenamiento supervisado sobre pares.

Dos detalles:

- La pérdida le da **más peso a los pares que el modelo todavía ordena mal**, como un clasificador. Por eso las curvas se separan rápido al principio y después se aplanan.
- DPO es **offline**: solo aprende de las respuestas que están en los pares. No prueba respuestas nuevas, a diferencia de PPO o GRPO.

¿Cambió su forma de responder?

In [ ]:
politica_dpo = entrenador_dpo.model.eval()
for q, a in zip(PREGUNTAS_PRUEBA, antes):
    print(f"{q}\n   antes:   {a[:150]}…\n   después: {responder(politica_dpo, q)}\n")

**🎛️ Su turno.** Háganle al modelo una pregunta propia y comparen las dos respuestas. Prueben con algo que la tienda no hace, o con una pregunta que no tenga que ver con una tienda.

In [ ]:
#@title 🎛️ Su pregunta, antes y después de DPO { run: "auto" }
pregunta_dpo = "¿Tienen despacho el mismo día?" #@param {type:"string"}

with politica_dpo.disable_adapter():           # con el adapter apagado es el modelo original: la referencia
    print("antes:  ", responder(politica_dpo, pregunta_dpo)[:300])
print("\ndespués:", responder(politica_dpo, pregunta_dpo))

Con 60 pares y menos de un minuto de entrenamiento, el modelo adoptó la voz de la tienda en preguntas que no estaban en los datos. Lean con cuidado las respuestas: el **estilo** es el que pedían las preferencias, pero los **hechos** pueden estar inventados. DPO enseña qué tipo de respuesta se prefiere; no le da al modelo información que no tenía.

**Para discutir:** las elegidas siempre decían que sí. ¿Qué creen que responde ahora el modelo si le preguntan por algo que la tienda no hace?

## 2.2 La recompensa implícita de un LLM real

DPO dice que un modelo alineado lleva una recompensa escondida:

$$r(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{ref}(y \mid x)}.$$

Podemos medirla en un par real: **Qwen2.5-0.5B-Instruct** (alineado, con SFT y aprendizaje por preferencias) contra **Qwen2.5-0.5B** (el modelo base del que partió). Si el post-entrenamiento funcionó, la recompensa implícita debería ser mayor para la respuesta que una persona preferiría.

La primera vez descarga ~2 GB.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
qwen_base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", dtype=torch.float32).eval()
qwen_instruct = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32).eval()

def logprob_respuesta(modelo, pregunta, respuesta):
    """log p(respuesta | pregunta): suma sobre los tokens de la respuesta, con la plantilla de chat."""
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": pregunta}], tokenize=False, add_generation_prompt=True)
    n_prompt = len(tokenizer(prompt).input_ids)
    ids = tokenizer(prompt + respuesta + "<|im_end|>", return_tensors="pt").input_ids
    with torch.no_grad():
        lp = torch.log_softmax(modelo(ids).logits[0, :-1], -1)
    objetivo = ids[0, 1:]
    return lp[n_prompt - 1:].gather(-1, objetivo[n_prompt - 1:, None]).sum().item()

In [ ]:
PARES = [
    ("¿Hacen envíos a regiones?",
     "Sí, enviamos a todo Chile. A regiones el despacho tarda entre 3 y 5 días hábiles.", "Envíos."),
    ("¿Cuál es la capital de Francia?",
     "La capital de Francia es París.", "La respuesta depende de cómo se defina la palabra capital."),
    ("Mi pedido llegó dañado, ¿qué hago?",
     "Lamento lo ocurrido. Envíanos una foto del producto y tu número de pedido, y gestionamos el cambio.",
     "No es nuestro problema, reclame a la empresa de transporte."),
    ("Escribe un saludo breve para un cliente.",
     "¡Hola! Gracias por escribirnos, ¿en qué te podemos ayudar?", "Qué quiere."),
    ("Resume en una frase: la reunión se movió del martes al jueves en la tarde.",
     "La reunión ahora es el jueves en la tarde.", "La reunión es el martes."),
    ("¿Cuánto es 15 por 4?", "15 por 4 es 60.", "15 por 4 es 45."),
]

beta = 0.1
print(" r(y_w)   r(y_l)   ¿acierta?   pregunta")
for pregunta_p, y_w, y_l in PARES:
    r_w = beta * (logprob_respuesta(qwen_instruct, pregunta_p, y_w) - logprob_respuesta(qwen_base, pregunta_p, y_w))
    r_l = beta * (logprob_respuesta(qwen_instruct, pregunta_p, y_l) - logprob_respuesta(qwen_base, pregunta_p, y_l))
    print(f"{r_w:+6.2f}   {r_l:+6.2f}      {'sí' if r_w > r_l else 'no':3s}     {pregunta_p}")

**🎛️ Su turno.** Escriban una pregunta y dos respuestas. ¿Cuál prefiere la recompensa implícita? Prueben con dos respuestas igual de correctas, pero con distinto tono o largo.

In [ ]:
#@title 🎛️ Su propio par { run: "auto" }
pregunta_par = "¿A qué hora abren?" #@param {type:"string"}
respuesta_a = "Abrimos de lunes a viernes, de 9 a 18 horas." #@param {type:"string"}
respuesta_b = "De 9 a 18." #@param {type:"string"}

def recompensa_implicita(pregunta, respuesta, beta=0.1):
    return beta * (logprob_respuesta(qwen_instruct, pregunta, respuesta) - logprob_respuesta(qwen_base, pregunta, respuesta))

for nombre, respuesta in [("A", respuesta_a), ("B", respuesta_b)]:
    print(f"{nombre}: {recompensa_implicita(pregunta_par, respuesta):+6.2f}   {respuesta}")

La recompensa implícita ordena bien los seis pares, pero fíjense **dónde** la diferencia es grande y dónde es mínima:

- Grande en tono y utilidad: el saludo cortante, el "Envíos." seco o el rechazo al cliente.
- Casi nula en hechos: "15 por 4 es 45" o la reunión el martes.

El post-entrenamiento cambió sobre todo el **estilo**; distinguir lo verdadero de lo falso ya dependía de lo que sabía el modelo base. (Ojo: Qwen no se entrenó con DPO puro; aquí usamos la fórmula de DPO como lente para mirar qué cambió entre ambos modelos.)

# 3. GRPO y recompensas verificables

> **Datos:** prompts + un verificador (**RLVR**). Nadie escribe respuestas correctas ni compara respuestas.

**GRPO es PPO sin crítico.** El ciclo es el mismo (generar, poner nota, calcular la ventaja, actualizar con recorte), con dos cambios ligados: se generan $G$ respuestas por prompt en vez de una, y lo esperado ya no lo estima un segundo LLM, sino el promedio del grupo. Es la línea base de la clase (restar lo esperado), calculada con las respuestas al mismo prompt.

1. Para cada problema se generan $G$ respuestas.
2. Se les pone nota.
3. La ventaja de cada respuesta es su recompensa relativa al grupo: $\hat A_i = (r_i - \text{promedio}) / \text{desv}$. Todos sus tokens comparten esa ventaja.

La nota puede venir de un modelo de recompensa o de un verificador: eso no depende de GRPO. Aquí usamos un verificador, porque en matemática basta comparar el número final con la solución.

Nuestra tarea: sumas de dos números, respondiendo **solo con el número**. El verificador es una regla de una línea: 1 si la respuesta es exactamente el número correcto, 0 si no.

**Antes de ejecutar:** Qwen2.5-0.5B es un modelo chico. ¿Qué fracción de estas sumas creen que acierta?

In [ ]:
#@title 🎛️ Anoten su predicción antes de ejecutar: ¿qué porcentaje de las sumas acierta Qwen?
mi_prediccion = 80 #@param {type:"slider", min:0, max:100, step:5}

# Cada problema: el prompt en formato de chat y la solución, que solo verá el verificador
azar = random.Random(0)
def problema():
    a, b = azar.randint(11, 59), azar.randint(11, 39)
    return {"prompt": [{"role": "user", "content": f"¿Cuánto es {a} + {b}? Responde solo con el número."}], "solucion": str(a + b)}

problemas = Dataset.from_list([problema() for _ in range(320)])      # para entrenar
problemas_prueba = [problema() for _ in range(20)]                   # para medir antes y después

def nota(texto, solucion):
    """El verificador: 1 si la respuesta es exactamente el número correcto."""
    return float(texto.strip().rstrip(".") == solucion)

def verificador_sumas(completions, solucion, **kwargs):
    """La función de recompensa en el formato de TRL: recibe las respuestas generadas y las columnas extra del dataset."""
    # cada respuesta llega como una lista con un mensaje: [{"role": "assistant", "content": "82"}]
    return [nota(c[0]["content"], s) for c, s in zip(completions, solucion)]

tokenizer.padding_side = "left"       # para generar por lotes
modelo_grpo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(DISPOSITIVO)

@torch.no_grad()
def muestras(modelo, k=8):
    """k respuestas por problema de prueba, con temperatura 1. Devuelve una matriz de notas (problemas × k) y los textos."""
    entrada = tokenizer.apply_chat_template([p["prompt"] for p in problemas_prueba], add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True, padding=True).to(modelo.device)
    torch.manual_seed(0)
    # do_sample=True con temperatura 1: se muestrea, así que las k respuestas a un problema pueden ser distintas
    salida = modelo.generate(**entrada, max_new_tokens=16, do_sample=True, temperature=1.0, num_return_sequences=k)
    textos = tokenizer.batch_decode(salida[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True)
    # las respuestas vienen en orden: las k primeras son del problema 0, las k siguientes del 1, etc.
    notas = torch.tensor([nota(t, problemas_prueba[i // k]["solucion"]) for i, t in enumerate(textos)]).view(-1, k)
    return notas, [textos[i:i + k] for i in range(0, len(textos), k)]

notas_antes, textos_antes = muestras(modelo_grpo)
print(f"Acierto con una muestra:             {notas_antes.mean():.0%}   (su predicción: {mi_prediccion} %)")
print(f"Acierto con la mejor de 8 (pass@8):  {(notas_antes.max(1).values > 0).float().mean():.0%}\n")
print("solución   respuestas del grupo                               ventajas de GRPO")
for p, textos, notas in list(zip(problemas_prueba, textos_antes, notas_antes))[:6]:
    ventaja = (notas - notas.mean()) / (notas.std() + 1e-4)      # la ventaja de grupo, como la calcula TRL
    print(f"{p['solucion']:>8}   {str([t.strip() for t in textos]):50s}  {[round(x, 1) for x in ventaja.tolist()]}")

Tres cosas para mirar:

- **Best-of-N** sube mucho el acierto respecto de una sola muestra (es el pass@k de las diapositivas). Con un verificador, muestrear varias veces ya es una mejora: muchas veces la respuesta correcta está entre las que el modelo sabe generar.
- Donde el grupo **mezcla** aciertos y errores, GRPO da ventaja positiva a las correctas y negativa a las incorrectas.
- Donde **todas** aciertan o todas fallan, las ventajas son 0: ese problema no aporta señal.

Es la diapositiva “GRPO en el laboratorio: ¿qué grupos enseñan?”: **solo los grupos mezclados enseñan**. Por eso el RLVR necesita problemas con la dificultad justa para el modelo, difíciles pero no imposibles. Fíjense también en el tamaño de las ventajas: un único acierto en un grupo de 8 recibe +2,5, mucho más que cada uno de los errores.

Ahora entrenemos con `GRPOTrainer`. La recompensa es nuestra función de Python; no hay respuestas correctas escritas por nadie en los datos de entrenamiento que ve el modelo, solo la nota del verificador.

In [ ]:
config = GRPOConfig(
    output_dir="grpo",                 # carpeta de trabajo del entrenador
    max_steps=40,                      # 40 pasos de optimización
    per_device_train_batch_size=16,    # respuestas generadas por paso
    num_generations=8,                 # el grupo: 8 respuestas por problema (16 / 8 = 2 problemas por paso)
    max_completion_length=16,          # largo máximo de cada respuesta, en tokens
    learning_rate=1e-4,                # tamaño de cada actualización de los pesos del adapter
    beta=0.0,                          # peso del freno KL: 0 lo apaga, y TRL no carga el modelo de referencia
    logging_steps=4,                   # registra la recompensa promedio cada 4 pasos
    save_strategy="no",                # no guarda puntos de control en disco
    report_to="none",                  # no envía los registros a servicios externos (como Weights & Biases)
    disable_tqdm=True,                 # sin barra de progreso: el avance lo muestra entrenar()
    seed=0,                            # semilla, para que el resultado se pueda repetir
)
entrenador_grpo = GRPOTrainer(
    model=modelo_grpo,                 # el LLM que se entrena (la política), ya cargado
    reward_funcs=[verificador_sumas],  # la recompensa es nuestra función de Python; puede haber varias, y se suman
    args=config,                       # la configuración de arriba
    train_dataset=problemas,           # columna "prompt"; las demás ("solucion") llegan al verificador
    processing_class=tokenizer,        # el tokenizador: arma el prompt con la plantilla de chat y decodifica las respuestas
    peft_config=lora("CAUSAL_LM"),     # LoRA: el modelo queda congelado y solo se entrena el adapter
)
entrenar(entrenador_grpo)              # en cada paso: genera los grupos, les pone nota, calcula las ventajas y actualiza

# "reward": la nota promedio de las respuestas generadas en esos pasos
pasos, recompensa = registros(entrenador_grpo, "reward")
plt.figure(figsize=(6, 3))
plt.plot(pasos, recompensa, color=MORADO, lw=2)
plt.ylim(0, 1.05); plt.xlabel("paso de entrenamiento"); plt.ylabel("recompensa promedio")
plt.tight_layout(); plt.show()

In [ ]:
# Las mismas 20 sumas de prueba y la misma semilla que antes de entrenar
notas_despues, textos_despues = muestras(entrenador_grpo.model.eval())
print("                          antes   después")
print(f"Acierto con una muestra:  {notas_antes.mean():5.0%}   {notas_despues.mean():5.0%}")
print(f"Acierto con pass@8:       {(notas_antes.max(1).values > 0).float().mean():5.0%}   {(notas_despues.max(1).values > 0).float().mean():5.0%}")

plt.figure(figsize=(4.5, 3))
plt.bar(["antes", "después"], [notas_antes.mean(), notas_despues.mean()], color=[GRIS, MORADO])
plt.ylim(0, 1.05); plt.ylabel("acierto con una muestra"); plt.tight_layout(); plt.show()

**🎛️ Su turno.** Elijan una suma y miren ocho respuestas del modelo antes y después de GRPO. Prueben con números más grandes que los del entrenamiento (que iban de 11 a 59): ¿lo aprendido se mantiene?

In [ ]:
#@title 🎛️ Su propia suma: 8 respuestas antes y después de GRPO { run: "auto" }
sumando_1 = 47 #@param {type:"integer"}
sumando_2 = 38 #@param {type:"integer"}

modelo_final = entrenador_grpo.model.eval()
entrada = tokenizer.apply_chat_template([{"role": "user", "content": f"¿Cuánto es {sumando_1} + {sumando_2}? Responde solo con el número."}],
                                        add_generation_prompt=True, return_tensors="pt", return_dict=True).to(modelo_final.device)

@torch.no_grad()
def ocho_respuestas():
    torch.manual_seed(0)
    salida = modelo_final.generate(**entrada, max_new_tokens=16, do_sample=True, temperature=1.0, num_return_sequences=8)
    return [t.strip() for t in tokenizer.batch_decode(salida[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True)]

with modelo_final.disable_adapter():           # con el adapter apagado: el modelo antes de entrenar
    respuestas_antes = ocho_respuestas()
respuestas_despues = ocho_respuestas()

solucion = str(sumando_1 + sumando_2)
print(f"{sumando_1} + {sumando_2} = {solucion}\n")
for nombre, respuestas in [("antes", respuestas_antes), ("después", respuestas_despues)]:
    print(f"{nombre:8s} {respuestas}   aciertos: {sum(nota(t, solucion) for t in respuestas):.0f} de 8")

El modelo mejoró en sumas que no vio, y nadie le mostró una sola respuesta correcta: generó, el verificador puso notas y GRPO subió la probabilidad de lo que funcionó. Es la receta de DeepSeek-R1 en miniatura.

Dos advertencias:

- La curva de recompensa es **ruidosa**: cada punto es el promedio de pocos grupos. En RL es normal; por eso se mira la tendencia y se evalúa aparte.
- El verificador también premia el **formato** (solo el número). Si la regla tiene fallas, el modelo las puede explotar igual que con un modelo de recompensa: el reward hacking no desaparece del todo.

Fíjense en `beta=0.0`: aquí entrenamos sin el freno KL, como hace por defecto TRL para GRPO. Con un verificador estricto el riesgo de alejarse de la referencia es menor que con un modelo de recompensa aprendido, como veremos en la sección 4.

# 4. RLHF de verdad: reward hacking y el freno KL

> **Datos:** prompts + el modelo de recompensa de la sección 1.2 (**RLHF**).

En la sección 3 la nota la ponía un verificador. Ahora la pondrá el **modelo de recompensa** r(x, y) que entrenamos en la sección 1.2 con pares de preferencias: eso es RLHF. El algoritmo es el mismo GRPO; lo único que cambia es la función de recompensa.

Recuerden lo que vimos de ese juez: ordenaba bien los pares de prueba, pero también le ponía buena nota a una respuesta larga y vacía.

**Antes de ejecutar:** si el modelo se entrena para sacar la mejor nota posible de ese juez, ¿qué creen que va a aprender?

In [ ]:
@torch.no_grad()
def juez(prompts, completions, **kwargs):
    """La función de recompensa: la nota del modelo de recompensa de la sección 1.2 para cada respuesta generada."""
    textos = [tok_rm.apply_chat_template(p + c, tokenize=False) for p, c in zip(prompts, completions)]   # prompt + respuesta, en formato de chat
    entrada = tok_rm(textos, return_tensors="pt", padding=True).to(modelo_rm.device)
    return modelo_rm(**entrada).logits[:, 0].tolist()

# Solo prompts: las preguntas de la tienda, sin ninguna respuesta
preguntas_tienda = Dataset.from_list([{"prompt": f["prompt"]} for f in preferencias] * 2)

def entrenar_rlhf(beta):
    """GRPO sobre un Qwen nuevo, con el modelo de recompensa como juez. beta es el peso del freno KL."""
    config = GRPOConfig(
        output_dir="rlhf",                 # carpeta de trabajo del entrenador
        max_steps=30,                      # 30 pasos de optimización
        per_device_train_batch_size=16,    # respuestas generadas por paso
        num_generations=8,                 # 8 respuestas por pregunta, 2 preguntas por paso
        max_completion_length=60,          # largo máximo de cada respuesta, en tokens
        learning_rate=1e-4,                # tamaño de cada actualización de los pesos del adapter
        beta=beta,                         # 0: sin freno. Mayor que 0: alejarse de la referencia cuesta
        logging_steps=3,                   # registra la nota promedio del juez cada 3 pasos
        save_strategy="no",                # no guarda puntos de control en disco
        report_to="none",                  # no envía los registros a servicios externos
        disable_tqdm=True,                 # sin barra de progreso: el avance lo muestra entrenar()
        seed=0,                            # misma semilla con y sin freno: la comparación es justa
    )
    entrenador = GRPOTrainer(
        # un Qwen recién cargado en cada llamada, para que los dos entrenamientos partan del mismo modelo
        model=AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(DISPOSITIVO),
        reward_funcs=[juez],               # lo único que cambia respecto de la sección 3
        args=config,                       # la configuración de arriba
        train_dataset=preguntas_tienda,    # solo la columna "prompt": no hay respuestas ni soluciones
        processing_class=tokenizer,        # el tokenizador: arma el prompt y decodifica las respuestas
        peft_config=lora("CAUSAL_LM"),     # LoRA: con beta > 0, la referencia es el modelo con el adapter apagado
    )
    entrenar(entrenador)
    return entrenador

def mostrar(respuestas, nombre):
    """Las respuestas a las preguntas de prueba, con la nota que les pone el juez."""
    for q, respuesta in zip(PREGUNTAS_PRUEBA, respuestas):
        nota_juez = juez([[{"role": "user", "content": q}]], [[{"role": "assistant", "content": respuesta}]])[0]
        print(f"{nombre}  r = {nota_juez:+5.1f}   {q}\n      → {respuesta[:200]}\n")

entrenador_sin_freno = entrenar_rlhf(beta=0.0)
pasos_sin, r_sin = registros(entrenador_sin_freno, "reward")

plt.figure(figsize=(6, 3))
plt.plot(pasos_sin, r_sin, color=ROJO, lw=2)
plt.xlabel("paso de entrenamiento"); plt.ylabel("nota promedio del juez")
plt.tight_layout(); plt.show()

mostrar(antes, "antes  ")           # las respuestas del modelo original, de la sección 2.1
mostrar([responder(entrenador_sin_freno.model.eval(), q) for q in PREGUNTAS_PRUEBA], "después")

La nota del juez sube: desde su punto de vista, el entrenamiento fue un éxito. Ahora lean las respuestas.

El modelo original decía, con razón, que es una inteligencia artificial y que no hace envíos. El modelo entrenado aprendió lo que al juez le gusta: responder que sí, con seguridad y en el tono de una tienda. Para lograrlo **inventa** los datos: no sabe nada de la tienda. El juez premia el tono, no la verdad, porque en sus datos de entrenamiento todas las respuestas elegidas eran afirmativas y amables.

Eso es **reward hacking**: la medida sube y lo que queríamos medir, no. Es la ley de Goodhart de la clase.

**🎛️ Su turno: el freno.** El remedio que vimos es penalizar que el modelo se aleje de la referencia, con el peso β. Elijan un valor y vuelvan a entrenar (tarda lo mismo que la celda anterior).

In [ ]:
#@title 🎛️ Elijan β y entrenen con freno
beta_freno = 0.5 #@param {type:"slider", min:0, max:2, step:0.1}

entrenador_con_freno = entrenar_rlhf(beta=beta_freno)
pasos_con, r_con = registros(entrenador_con_freno, "reward")

plt.figure(figsize=(6, 3))
plt.plot(pasos_sin, r_sin, color=ROJO, lw=2, label="sin freno (β = 0)")
plt.plot(pasos_con, r_con, color=AZUL, lw=2, label=f"con freno (β = {beta_freno})")
plt.xlabel("paso de entrenamiento"); plt.ylabel("nota promedio del juez")
plt.legend(frameon=False); plt.tight_layout(); plt.show()

mostrar([responder(entrenador_con_freno.model.eval(), q) for q in PREGUNTAS_PRUEBA], "con freno")

Con el freno, la nota sube menos y las respuestas se parecen mucho más a las del modelo original: alejarse de la referencia cuesta, así que al modelo ya no le conviene cambiar tanto solo para agradar al juez.

β es una perilla, no una solución: con β grande el modelo casi no aprende; con β chico vuelve el reward hacking. El arreglo de fondo está en el juez.

**Para discutir:** ¿qué pares de preferencias agregarían a los datos de la sección 1.2 para que el juez deje de premiar respuestas afirmativas e inventadas?

### ¿Y PPO?

TRL también trae PPO (`from trl.experimental.ppo import PPOConfig, PPOTrainer`), pero no lo ejecutamos aquí: necesita cuatro modelos a la vez (política, referencia, modelo de recompensa y crítico) y es más lento y delicado de ajustar. 

# Resumen

| Sección | Lo que vimos | Herramienta |
|---|---|---|
| 1 | Una función de recompensa r(x, y) se aprende con comparaciones; solo importan las diferencias. El modelo aprendido tiene atajos. | PyTorch y `RewardTrainer` |
| 2 | DPO alinea con una pérdida supervisada sobre pares; un modelo alineado lleva una recompensa implícita. | `DPOTrainer` |
| 3 | Con un verificador no hace falta modelo de recompensa ni crítico: best-of-N y las ventajas de grupo de GRPO. | `GRPOTrainer` |
| 4 | Con un modelo de recompensa como juez, el modelo aprende a agradarlo (reward hacking); la penalización KL lo frena. | `GRPOTrainer` |

# Ejercicios

1. **Más o menos comparaciones.** Repitan la sección 1.1 con 100 y con 30.000 comparaciones. ¿Cuántas hacen falta para ordenar bien las cinco respuestas?
2. **Engañar al juez.** Con el formulario de la sección 1.2, escriban una respuesta que no conteste la pregunta y saque más nota que la respuesta útil. ¿Qué pares agregarían a los datos para corregirlo?
3. **DPO más o menos conservador.** En la sección 2.1, cambien `beta` a 0,01 y a 1. ¿Cómo cambian la curva de recompensa implícita y las respuestas?
4. **El precio del freno.** Con el formulario de la sección 4, prueben β = 0,1 y β = 2. ¿Qué pasa con la nota del juez y con las respuestas en cada caso?
5. **Otro verificador.** En la sección 3, cambien `nota` para aceptar también respuestas como “42.” o “Son 42”. ¿Cambia lo que aprende el modelo? ¿Y si la regla premia cualquier respuesta que contenga un número?